# DarkSight — E2 & E3 on Kaggle (run all, ~1.5–2 h)

**E2** = YOLOv8s fine-tuned on raw ExDark, tested on raw → the baseline.
**E3** = the *same* detector, tested on Zero-DCE enhanced images → enhancement applied at test time only.
Both come from one training run, so this notebook trains once.

### Before you press Run All
Right-hand panel → **Accelerator: GPU T4 x2** · **Internet: On** · **Persistence: Files and Variables**.
Nothing else to set: if no ExDark dataset is attached, the notebook downloads it itself.

Everything is written to `/kaggle/working` (survives restarts, downloadable); the big intermediate
datasets go to `/kaggle/temp`. Repo: https://github.com/Manideep-21/darksight.git

In [ ]:
# 1 · environment
import os, subprocess, time
T0 = time.time()
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
os.makedirs('/kaggle/temp', exist_ok=True)
%cd /kaggle/working
!git clone -q https://github.com/Manideep-21/darksight.git darksight 2>/dev/null || (cd darksight && git pull -q)
%cd /kaggle/working/darksight
!pip -q install "ultralytics>=8.3,<9" scikit-image gdown
os.environ['DARKSIGHT_DATA'] = '/kaggle/temp/exdark_yolo'
import torch, ultralytics
print('torch', torch.__version__, '| ultralytics', ultralytics.__version__, '|', torch.cuda.device_count(), 'GPU(s)')

In [ ]:
# 2 · COCO-pretrained checkpoint (GitHub release, with a Hugging Face fallback)
from pathlib import Path
for m in ['yolov8s.pt', 'yolov8n.pt']:
    p = Path('weights') / m
    if not p.exists():
        !wget -q -O {p} https://github.com/ultralytics/assets/releases/download/v8.4.0/{m} || \
         wget -q -O {p} https://huggingface.co/Ultralytics/YOLOv8/resolve/main/{m}
    print(m, p.stat().st_size // 1024, 'KB')

In [ ]:
# 3 · locate ExDark; download it if nothing usable is attached
ok = subprocess.run(['python', 'scripts/find_exdark.py', '/kaggle/input']).returncode == 0
if not ok:
    print('\nNo complete ExDark in /kaggle/input — downloading the official copy (~1.5 GB)...')
    !mkdir -p /kaggle/temp/src
    !gdown -q 1BHmPgu8EsHoFDDkMGLVoXIlCth2dW6Yx -O /kaggle/temp/ExDark.zip
    !gdown -q 1P3iO3UYn7KoBi5jiUkogJq96N6maZS1i -O /kaggle/temp/ExDark_Annno.zip
    !unzip -q -o /kaggle/temp/ExDark.zip -d /kaggle/temp/src && unzip -q -o /kaggle/temp/ExDark_Annno.zip -d /kaggle/temp/src
    !rm -rf /kaggle/temp/src/__MACOSX
    SEARCH = '/kaggle/temp/src'
    !python scripts/find_exdark.py {SEARCH}
else:
    SEARCH = '/kaggle/input'
IMAGES = !python scripts/find_exdark.py {SEARCH} --print images
ANNOTS = !python scripts/find_exdark.py {SEARCH} --print annotations
IMAGES, ANNOTS = IMAGES[-1].strip(), ANNOTS[-1].strip()
print('images     :', IMAGES)
print('annotations:', ANNOTS)

In [ ]:
# 4 · convert to YOLO format — official split 3000 train / 1800 val / 2563 test  (~5-8 min)
!python scripts/prepare_exdark.py --images "{IMAGES}" --annotations "{ANNOTS}" \
    --out /kaggle/temp/exdark_yolo/raw --meta data/exdark_meta.csv \
    --report /kaggle/working/data_stats.md --workers 4

In [ ]:
# 5 · eyeball 3 label overlays — boxes MUST sit on the objects before we train anything
!python scripts/visualize_labels.py --root /kaggle/temp/exdark_yolo/raw --split train --n 3 --out /kaggle/working/label_check
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/kaggle/working/label_check/train/*.png'))[:3]:
    display(Image(p, width=420))

In [ ]:
# 6 · Zero-DCE enhanced copy of the TEST split — this is what E3 is evaluated on  (~3 min)
!python scripts/enhance_dataset.py --src /kaggle/temp/exdark_yolo/raw --dst /kaggle/temp/exdark_yolo/enhanced \
    --splits test --half
!python scripts/verify_dataset.py --a /kaggle/temp/exdark_yolo/raw --b /kaggle/temp/exdark_yolo/enhanced --splits test

In [ ]:
# 7 · smoke train (~3 min) — catches data/DDP problems before the real run
!python scripts/train_detector.py --dataset /kaggle/temp/exdark_yolo/raw --name smoke \
    --model yolov8n.pt --epochs 1 --fraction 0.1 --device 0,1 --batch 32 --project /kaggle/temp/runs_smoke

In [ ]:
# 8 · TRAIN det_raw — yolov8s, 60 epochs, configs/train.yaml unchanged   (~45-75 min on 2x T4)
# --batch 32 on 2 GPUs = 16 images per GPU, i.e. the same per-GPU batch as the single-GPU config.
# Use these exact flags again for det_enh later, or E2 vs E4 stops being a fair comparison.
!python scripts/train_detector.py --dataset /kaggle/temp/exdark_yolo/raw --name det_raw \
    --device 0,1 --batch 32 --project /kaggle/working/runs
!cp weights/det_raw.pt /kaggle/working/det_raw.pt
print('checkpoint saved to /kaggle/working/det_raw.pt')

**If the session died during cell 8:** re-run cells 1–6 (~15 min), then run
`!python scripts/train_detector.py --resume /kaggle/working/runs/det_raw/weights/last.pt`
in a new cell. `save_period: 5` means at most 5 epochs are lost.

In [ ]:
# 9 · E2 (raw) and E3 (enhanced) on the 2563 test images   (~6 min)
!python scripts/run_experiment.py --exp E2 E3 --device 0

In [ ]:
# 10 · is the difference real? paired bootstrap + per-lighting breakdown + tables
!python scripts/compare_experiments.py --a E2 --b E3 --n 1000
!python scripts/eval_by_condition.py --exp E2 E3
!python scripts/make_results_table.py

In [ ]:
# 11 · figures + detection images for the report
!python scripts/make_qualitative.py --raw-exp E2 --enh-exp E3 --pick gain --n 6
!python scripts/make_qualitative.py --raw-exp E2 --enh-exp E3 --pick loss --n 3
!python scripts/render_predictions.py --exp E2 E3 --side-by-side E2 E3 --conf 0.25

In [ ]:
# 12 · results
from IPython.display import Markdown, Image, display
display(Markdown(open('results/summary_test.md').read()))
for p in ['results/figures/overall_test.png', 'results/figures/per_class_test.png', 'results/figures/by_light_test.png']:
    display(Image(p, width=780))

In [ ]:
# 13 · package everything to download from the Output panel
!cd /kaggle/working/darksight && zip -q -r /kaggle/working/darksight_E2_E3.zip \
    results reports data/exdark_meta.csv -x "results/E*/predictions_*.jsonl"
!du -h /kaggle/working/darksight_E2_E3.zip /kaggle/working/det_raw.pt
print(f'total notebook time: {(time.time()-T0)/60:.0f} min')
print('''
Download from the Output panel:
  det_raw.pt              -> put in weights/ locally, then: python app/app.py
  darksight_E2_E3.zip     -> metrics, figures, detection images; commit the JSON/CSV to the repo

Next: train det_enh on the enhanced TRAIN split to get E4 (the proposed pipeline).
  !python scripts/enhance_dataset.py --src /kaggle/temp/exdark_yolo/raw --dst /kaggle/temp/exdark_yolo/enhanced --splits train val --half
  !python scripts/train_detector.py --dataset /kaggle/temp/exdark_yolo/enhanced --name det_enh --device 0,1 --batch 32 --project /kaggle/working/runs
  !python scripts/run_experiment.py --exp E4 --device 0 && python scripts/compare_experiments.py --a E2 --b E4
''')